In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar
import pandas as pd
import cartopy.util
import xesmf as xe
import src.XRO_utils
import cartopy.crs as ccrs

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(r"/glade/work/kcarr/lme_data")

## Funcs

In [ ]:
def prep_seasonal(y):
    """prepare data for EOF analysis"""

    ## resample to seasonal
    y_ = y.resample({"time": "QS-JAN"}).mean()

    ## get year start
    yrs = y_.time.dt.year.values
    mths = y_.time.dt.month.values
    year_start = np.array([y - 1 if (m < 7) else y for (y, m) in zip(yrs, mths)])

    ## get season
    season_dict = {1: "JFM", 4: "AMJ", 7: "JAS", 10: "OND"}
    season = np.array([season_dict[m] for m in mths])

    ## create multi-index
    new_idx = pd.MultiIndex.from_arrays([year_start, season], names=["y0", "season"])

    ## convert to xr
    new_idx_xr = xr.Coordinates.from_pandas_multiindex(new_idx, dim="time")

    ## add to original data
    y_ = y_.assign_coords(new_idx_xr).unstack("time")

    ## re-order seasons
    y_ = y_.reindex({"season": ["JAS", "OND", "JFM", "AMJ"]})

    return y_.isel(y0=slice(1, -1))


def prep_and_trim(x):
    """rename coordinates and trim in time"""

    # ## time coordinates for trimming
    time_idx = dict(time=slice("0900-01", "1849-12"))

    ## rename spatial coords
    coord_dict = dict(lat="latitude", lon="longitude")

    return x.sel(time_idx).rename(coord_dict)


def plot_sst2(ax, data, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance",
        levels=make_cb_range(dx, nlev),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def make_cb_range(dx, nlevs):
    amp = dx * nlevs - dx / 2

    lev1 = np.arange(0, amp + dx / 2, dx) + dx / 2
    lev0 = -lev1[::-1]
    return np.concatenate([lev0, lev1])


## specify lons/lats for E/W boxes
LONS_E = [240, 280]
LONS_W = [150, 190]


## funcs to plot boxes
def plot_wbox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_W, lats=[-5, 5], **kwargs)


def plot_ebox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_E, lats=[-5, 5], **kwargs)


def add_gridlines(axs):
    """func to add gridlines to axs object"""

    for ax in axs[:-1, 0]:
        ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[],
            ylocs=[-20, 0, 20],
        )
    for ax in axs[-1]:
        gl_ = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[150, -170, -120, -80],
            ylocs=[-20, 0, 20],
        )
        gl_.top_labels = False
    gl_.left_labels = False

    return


def plot_level(ax, data, level, ls="-", c="white"):
    """plot single level on hovmoller"""
    ax.contour(
        data.longitude,
        data.latitude,
        data,
        levels=[level],
        colors=c,
        linestyles=ls,
        zorder=10,
        transform=ccrs.PlateCarree(),
    )
    return


def regress_pinv(X, x_vars, y_var, dims=["time", "member"]):
    """do nonlinear regression"""

    ## prep data
    stack = lambda x: x.stack(s=dims)  # .transpose(..., "s")
    X_ = stack(X[x_vars].to_dataarray(dim="v")).transpose(..., "v", "s")
    Y_ = stack(X[y_var]).transpose(..., "s")

    ## build coordinates for new array
    coords = {}
    for d in Y_.dims:
        if d != "s":
            coords[d] = Y_[d]

    ## add "v"
    coords["v"] = X_.v

    ## empty array to hold results
    m = xr.DataArray(coords=coords, dims=list(coords))

    ## do regression
    X_pinv = np.linalg.pinv(X_.values)
    m.values = np.einsum("...i,...ij->...j", Y_.values, X_pinv)

    return m.rename({"v": "param"}).squeeze(drop=True)


def regress_pinv_bymonth(x, **kwargs):
    """regress by month"""
    return x.groupby("time.month").map(regress_pinv, **kwargs)

In [ ]:
# # DIMS = ["time","member"]
# # new_coords = {}
# # for k,v in idx.coords.items():
# #     if k!="s":
# #         new_coords[k] = v

# # list(new_coords)

# # xr.Coordinates(new_coords)

# # idx.coords["time"]

# DIMS = ["time","member"]
# new_coords = list(set(list(sst.dims))-set(DIMS))
# new_coords = {
# + ["v"]

# idx.dims


# idx.drop_vars("time").coords

# for c, v in idx.coords.items():
#     print(v)

# xr.Coordinates({c : v if c in ["time"] else None for c, v in idx.coords.items()})

# idx.coords

## Load data

#### CESM2

In [ ]:
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
sst = xr.open_dataset(
    DATA_FP / "snr" / "sst_split_deseason.nc", decode_times=time_coder
).compute()
pr = xr.open_dataset(DATA_FP / "snr" / "pr_split_deseason.nc", decode_times=time_coder)
pr = pr.assign_coords({"time": sst.time}).compute()

## trim
sst = prep_and_trim(sst)
pr = prep_and_trim(pr)

## regrid
GRID = pr.isel(member=0, time=0)


def regrid_like_pr(x):
    """regrid like precip data"""
    regridder = xe.Regridder(x, GRID, "bilinear")
    return regridder(x)


## regrid sst
sst = regrid_like_pr(sst)
sst = sst.where(np.abs(sst.latitude) < 31)

In [ ]:
## helper func to get clim
get_clim = lambda x: x.groupby("time.month").mean()

## clims/anomalies
clim = xr.merge(
    [get_clim(sst["forced"]).rename("sst"), get_clim(pr["forced"]).rename("pr")]
)

anom = xr.merge([sst["anom"].rename("sst"), pr["anom"].rename("pr")])

### Reference data

## Regression

### Specify indices

In [ ]:
## specify index fn
idx_fn = lambda x: src.utils.get_nino3(x).rename("T")
# idx_fn = lambda x: src.utils.get_nino34(x.sel(season="JFM"))

## Get index
idx = idx_fn(anom["sst"])
# idx_ref = idx_fn(anom_ref["sst"])

### Compute

In [ ]:
## merge data
regress_data = xr.merge(
    [anom, idx, (idx**2).rename("T2"), xr.ones_like(idx).rename("ones")]
)

## compute sst
m_sst = regress_pinv_bymonth(regress_data, x_vars=["T", "T2", "ones"], y_var="sst")
m_sst = m_sst.to_dataset("param")


## compute precip
m_pr = regress_pinv_bymonth(regress_data, x_vars=["T", "T2", "ones"], y_var="pr")
m_pr = m_pr.to_dataset("param")

### Spatial plot

How to interpret this: 
- epochs with stronger ENSO will look more like quadratic plot
- epochs with weaker ENSO will look more like "const" plot

In [ ]:
## specify how to select season
# sel = lambda x: x.mean("season")
sel = lambda x: x.mean("month")

## specify intervals
dxs = np.array(
    [
        [0.3, 0.5],
        [0.1, 0.25],
        [0.1, 0.25],
    ],
)

fig = plt.figure(figsize=(10, 7.5), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=30, lon_range=(60, 285))
axs = src.utils.subplots_with_proj(fig, nrows=3, ncols=2, format_func=format_func)


cp00 = plot_sst2(axs[0, 0], sel(m_sst["T"]), dx=dxs[0, 0])
cp00 = plot_sst2(axs[0, 1], -sel(m_pr["T"]), dx=dxs[0, 1])

cp10 = plot_sst2(axs[1, 0], sel(m_sst["T2"]), dx=dxs[1, 0])
cp10 = plot_sst2(axs[1, 1], -sel(m_pr["T2"]), dx=dxs[1, 1])

cp10 = plot_sst2(axs[2, 0], sel(m_sst["ones"]), dx=dxs[2, 0])
cp10 = plot_sst2(axs[2, 1], -sel(m_pr["ones"]), dx=dxs[2, 1])
# cp11 = plot_sst2(axs[2, 1], sel(comps_norm["sst_asym"]), dx=dxs[2, 1])

## add formatting to axs
for ax in axs.flatten():
    plot_wbox(ax, c="gray", lw=0.8)
    plot_ebox(ax, c="gray", lw=0.8)
    src.utils.plot_nino34_box(ax, c="w", ls="--", lw=0.8, alpha=0.75)

## label
axs[0, 0].set_title("(a) Lin. (SST)", loc="left")
axs[0, 1].set_title(r"(b) Lin. (pr)", loc="left")
axs[1, 0].set_title("(c) Quad. (SST)", loc="left")
axs[1, 1].set_title("(d) Quad. (pr)", loc="left")
axs[2, 0].set_title("(e) Const. (SST)", loc="left")
axs[2, 1].set_title("(f) Const. (pr)", loc="left")

## add labels
add_gridlines(axs)

## aspect
for ax in axs.flatten():
    ax.set_aspect("auto")

bbox = dict(boxstyle="round", facecolor="white", alpha=1)
legend_kwargs = dict(x=0.01, y=0.02, fontdict=dict(size=12, color="gray"), bbox=bbox)
for ax, dx in zip(axs.flatten(), dxs.flatten()):
    ax.text(s=f"Interval: {dx} K", transform=ax.transAxes, **legend_kwargs)
    src.utils.plot_box(ax=ax, lons=[65, 140], lats=[5, 35], ls="--", c="k", lw=1)


plt.show()

### Seasonal plot

In [ ]:
def prep_hov(x):
    """prep data for hovmoller plot"""

    ## average over longitudes in monsoon region
    x_ = x.sel(longitude=slice(60, 140)).mean("longitude")

    ## transpose for hovmoller
    x_ = x_.transpose("latitude", "month")

    ## roll coords(?)
    x_ = x_.roll({"month": 7}, roll_coords=False)

    ## get coords
    return x_.month, x_.latitude, x_


def format_hov_ax(ax):
    """format hovmoller ax"""
    ax.set_xticks([1, 7, 12], labels=["Jul", "Jan", "Jun"])
    ax.set_ylim([-30, 30])
    for t in [-5, 5]:
        ax.axhline(t, ls="--", c="gray", lw=1)

    return

#### Plot clim.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(9, 4), layout="constrained")

cp0 = axs[0].contourf(
    *prep_hov(clim["pr"]),
    levels=np.arange(0, 12, 1),
    cmap="cmo.rain",
    extend="both",
)

cp1 = axs[1].contourf(
    *prep_hov(clim["pr"].mean("month") * xr.ones_like(clim["pr"])),
    levels=np.arange(0, 12, 1),
    cmap="cmo.rain",
    extend="both",
)

cp2 = axs[2].contourf(
    *prep_hov(clim["pr"] - clim["pr"].mean("month")),
    levels=make_cb_range(dx=1, nlevs=5),
    cmap="cmo.balance_r",
    extend="both",
)

## format/label
for ax in axs:
    format_hov_ax(ax)
    ax.set_yticks([])

## colorbars
# fig.colorbar(cp0, ax=axs[0], ticks=[0, 12])
fig.colorbar(cp1, ax=axs[1], ticks=[0, 12])
fig.colorbar(cp2, ax=axs[2], ticks=[-4, 4], label=r"mm day$^{-1}$")

plt.show()

#### Plot regression coefs

##### precip

Note: El Niños tend to flatten monsoon seasonal cycle; La Niñas intensify it(?)

In [ ]:
fig, axs_ = plt.subplots(3, 3, figsize=(9, 12), layout="constrained")

for j, (coef, amp) in enumerate([("T", 0.4), ("T2", 0.1), ("ones", 0.1)]):

    ## get axes for plotting
    axs = axs_[j]

    ## specify shared args
    kwargs = dict(
        levels=make_cb_range(dx=amp, nlevs=5),
        cmap="cmo.balance_r",
        extend="both",
    )

    cp0 = axs[0].contourf(
        *prep_hov(m_pr[coef]),
        **kwargs,
    )

    cp1 = axs[1].contourf(
        *prep_hov(m_pr[coef].mean("month") * xr.ones_like(m_pr[coef])),
        **kwargs,
    )

    cp2 = axs[2].contourf(
        *prep_hov(m_pr[coef] - m_pr[coef].mean("month")),
        **kwargs,
    )

    ## open contour baseline
    axs[2].contour(
        *prep_hov(clim["pr"] - clim["pr"].mean("month")),
        levels=make_cb_range(dx=2, nlevs=5),
        colors="k",
        linewidths=1,
    )

    ## format/label
    for ax in axs:
        format_hov_ax(ax)
        ax.set_yticks([])

    ## colorbars
    # fig.colorbar(cp1, ax=axs[1], ticks=[-amp, amp])
    fig.colorbar(cp2, ax=axs[2], ticks=[-amp * 5, amp * 5], label=r"mm day$^{-1}$")

plt.show()

##### SST

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(9, 4), layout="constrained")

cp0 = axs[0].contourf(
    *prep_hov(clim["sst"]),
    levels=np.arange(20, 30, 1),
    cmap="cmo.thermal",
    extend="both",
)

cp1 = axs[1].contourf(
    *prep_hov(clim["sst"].mean("month") * xr.ones_like(clim["sst"])),
    levels=np.arange(20, 30, 1),
    cmap="cmo.thermal",
    extend="both",
)

cp2 = axs[2].contourf(
    *prep_hov(clim["sst"] - clim["sst"].mean("month")),
    levels=make_cb_range(dx=0.6, nlevs=5),
    cmap="cmo.balance",
    extend="both",
)

## format/label
for ax in axs:
    format_hov_ax(ax)
    ax.set_yticks([])
    ax.set_ylim([-25, 25])

## colorbars
# fig.colorbar(cp0, ax=axs[0], ticks=[0, 12])
fig.colorbar(cp1, ax=axs[1], ticks=[0, 12])
fig.colorbar(cp2, ax=axs[2], ticks=[-4, 4], label=r"mm day$^{-1}$")

plt.show()

In [ ]:
fig, axs_ = plt.subplots(3, 3, figsize=(9, 12), layout="constrained")

for j, (coef, amp) in enumerate([("T", 0.1), ("T2", 0.025), ("ones", 0.1)]):

    ## get axes for plotting
    axs = axs_[j]

    ## specify shared args
    kwargs = dict(
        levels=make_cb_range(dx=amp, nlevs=5),
        cmap="cmo.balance_r",
        extend="both",
    )

    cp0 = axs[0].contourf(
        *prep_hov(m_sst[coef]),
        **kwargs,
    )

    cp1 = axs[1].contourf(
        *prep_hov(m_sst[coef].mean("month") * xr.ones_like(m_sst[coef])),
        **kwargs,
    )

    cp2 = axs[2].contourf(
        *prep_hov(m_sst[coef] - m_sst[coef].mean("month")),
        **kwargs,
    )

    ## open contour baseline
    axs[2].contour(
        *prep_hov(clim["sst"] - clim["sst"].mean("month")),
        levels=make_cb_range(dx=1, nlevs=5),
        colors="k",
        linewidths=1,
    )

    ## format/label
    for ax in axs:
        format_hov_ax(ax)
        ax.set_yticks([])
        ax.set_ylim([-25, 25])

    ## colorbars
    # fig.colorbar(cp1, ax=axs[1], ticks=[-amp, amp])
    fig.colorbar(cp2, ax=axs[2], ticks=[-amp * 5, amp * 5], label=r"mm day$^{-1}$")

plt.show()

## Decadal analysis (rectification)

In [ ]:
## separate data into epochs
rect_data = src.utils.unstack_month_and_year(anom).rolling({"year": 31}, center=True)
rect_data = rect_data.construct(window_dim="sample", stride=15)

## trim to remove NaNs
rect_data = rect_data.isel(year=slice(1, None, 2))

## compute stats
mu = rect_data.mean("sample")
sigma = rect_data.std("sample")

check results make sense

In [ ]:
regress_data = xr.merge(
    [
        # src.utils.get_nino3(mu["sst"]).rename("T_3"),
        src.utils.get_nino3(sigma["sst"]).rename("T_3"),
        xr.ones_like(src.utils.get_nino3(sigma["sst"])).rename("ones"),
        mu,
    ]
)

## compute sst
m_sst = regress_pinv(
    regress_data, x_vars=["T_3", "ones"], y_var="sst", dims=["year", "member"]
)
m_sst = m_sst.to_dataset("param")

m_pr = regress_pinv(
    regress_data, x_vars=["T_3", "ones"], y_var="pr", dims=["year", "member"]
)
m_pr = m_pr.to_dataset("param")

In [ ]:
## specify how to select season
# sel = lambda x: x.mean("season")
sel = lambda x: x.mean("month")

## specify intervals
dxs = np.array(
    [
        [0.05, 0.5],
        [0.1, 0.25],
        [0.1, 0.25],
    ],
)

fig = plt.figure(figsize=(5, 5), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=30, lon_range=(60, 285))
axs = src.utils.subplots_with_proj(fig, nrows=2, ncols=1, format_func=format_func)


cp00 = plot_sst2(axs[0, 0], sel(m_sst["T_3"]), dx=0.15)
cp10 = plot_sst2(axs[1, 0], -sel(m_pr["T_3"]), dx=0.4)
# cp11 = plot_sst2(axs[2, 1], sel(comps_norm["sst_asym"]), dx=dxs[2, 1])

## add formatting to axs
for ax in axs.flatten():
    plot_wbox(ax, c="gray", lw=0.8)
    plot_ebox(ax, c="gray", lw=0.8)
    src.utils.plot_nino34_box(ax, c="w", ls="--", lw=0.8, alpha=0.75)

## label
axs[0, 0].set_title("(a) SST", loc="left")
axs[1, 0].set_title("(b) Precip", loc="left")

## add labels
add_gridlines(axs)

## aspect
for ax in axs.flatten():
    ax.set_aspect("auto")

bbox = dict(boxstyle="round", facecolor="white", alpha=1)
legend_kwargs = dict(x=0.01, y=0.02, fontdict=dict(size=12, color="gray"), bbox=bbox)
for ax, dx in zip(axs.flatten(), dxs.flatten()):
    ax.text(s=f"Interval: {dx} K", transform=ax.transAxes, **legend_kwargs)
    src.utils.plot_box(ax=ax, lons=[65, 140], lats=[5, 35], ls="--", c="k", lw=1)


plt.show()

### Now do hovmoller

Note: THIS is weakening of seasonal cycle/monsoon precip.

In [ ]:
fig, axs_ = plt.subplots(2, 3, figsize=(9, 8), layout="constrained")

for j, (coef, amp) in enumerate([("T_3", 0.3), ("ones", 0.3)]):

    ## get axes for plotting
    axs = axs_[j]

    ## specify shared args
    kwargs = dict(
        levels=make_cb_range(dx=amp, nlevs=5),
        cmap="cmo.balance_r",
        extend="both",
    )

    cp0 = axs[0].contourf(
        *prep_hov(m_pr[coef]),
        **kwargs,
    )

    cp1 = axs[1].contourf(
        *prep_hov(m_pr[coef].mean("month") * xr.ones_like(m_pr[coef])),
        **kwargs,
    )

    cp2 = axs[2].contourf(
        *prep_hov(m_pr[coef] - m_pr[coef].mean("month")),
        **kwargs,
    )

    ## open contour baseline
    axs[2].contour(
        *prep_hov(clim["pr"] - clim["pr"].mean("month")),
        levels=make_cb_range(dx=2, nlevs=5),
        colors="k",
        linewidths=1,
    )

    ## format/label
    for ax in axs:
        format_hov_ax(ax)
        ax.set_yticks([])

    ## colorbars
    # fig.colorbar(cp1, ax=axs[1], ticks=[-amp, amp])
    fig.colorbar(cp2, ax=axs[2], ticks=[-amp * 5, amp * 5], label=r"mm day$^{-1}$")

plt.show()